# Тема 5. Model Context Protocol (MCP) и распределённое подключение возможностей

**Модуль II · Пример 2 из 4**

В Теме 4 инструменты были «зашиты» в код агента. Это порождает **жёсткую связанность**:
- дублирование реализаций в каждом агенте;
- хрупкость при изменении внешних API;
- зависимость от формата конкретной модели.

**MCP (Model Context Protocol)** — открытый стандарт, который отвязывает инструменты от агента по схеме **клиент — сервер**:
- **Сервер** публикует возможности (инструменты, ресурсы) — здесь это функции магазина.
- **Клиент** (внутри хоста-агента) подключается к серверу и вызывает их.
- Общение идёт по **JSON-RPC**; клиент выполняет **discovery** (обнаружение) доступных инструментов и **согласование версий** протокола.

> Мы поднимаем настоящий MCP-сервер магазина в отдельном процессе и подключаемся к нему официальным Python-SDK `mcp` через транспорт **stdio**.

### Что вы увидите
1. Реализация MCP-сервера на `FastMCP` (инструменты магазина).
2. Клиент: подключение, `initialize` (согласование версий), `list_tools` (discovery), `call_tool`.
3. Мост «MCP → языковая модель»: агент вызывает инструменты, опубликованные сервером.
4. **Замена сервера** на расширенную версию **без изменения кода агента**.
5. Когда MCP — избыточное усложнение (overengineering).

## Шаг 0. Конфигурация модели

Как и в Теме 4 — клиент к OpenAI-совместимому API. Модель понадобится на шаге, где агент вызывает инструменты через MCP.

In [1]:
import os, sys, json, asyncio
from openai import OpenAI

API_KEY = os.environ.get("LLM_API_KEY", "YOUR_API_KEY")
client = OpenAI(api_key=API_KEY, base_url=os.environ.get("LLM_BASE_URL", "YOUR_API_BASE_URL"))
MODEL = "qwen/qwen3.7-max"
print("Клиент модели готов.")

Клиент модели готов.


## Шаг 1. MCP-сервер магазина

Сервер лежит в отдельном файле `mcp_shop_server.py` — он запускается как самостоятельный процесс. Библиотека **`FastMCP`** превращает обычные Python-функции в MCP-инструменты: декоратор **`@mcp.tool()`** регистрирует функцию, а её сигнатура и докстринг автоматически становятся описанием и схемой (аналогично тому, что в Теме 4 мы делали руками).

`mcp.run(transport="stdio")` запускает сервер, общающийся через стандартный ввод/вывод — так клиент сможет поднять его как подпроцесс. Посмотрим на исходный код сервера:

In [2]:
from pathlib import Path
print(Path("mcp_shop_server.py").read_text())

"""MCP-сервер магазина «ШопБот» (Тема 5).

Публикует инструменты магазина по протоколу MCP через транспорт stdio.
Запуск вручную:  python mcp_shop_server.py   (ждёт JSON-RPC на stdin).
Обычно запускается клиентом как подпроцесс — см. notebook 02_mcp_protocol.ipynb.
"""
import logging
logging.getLogger().setLevel(logging.WARNING)  # тише лог, чтобы не засорять вывод

import pandas as pd
from mcp.server.fastmcp import FastMCP

CATALOG = pd.read_csv("data/products.csv")
ORDERS = {
    "ORD-1001": {"sku": "85123A", "name": "White Hanging Heart T-Light Holder",
                 "quantity": 2, "status": "shipped"},
}

mcp = FastMCP("shop", log_level="WARNING")


@mcp.tool()
def search_products(query: str, max_price: float | None = None, limit: int = 5) -> list[dict]:
    """Найти товары по ключевому слову в названии (каталог на английском)."""
    df = CATALOG[CATALOG["name"].str.contains(query, case=False, na=False)]
    if max_price is not None:
        df = df[df["price"] <= max_price]
  

## Шаг 2. Клиент MCP: подключение, discovery, вызов

Клиент поднимает сервер как подпроцесс (`StdioServerParameters`) и открывает сессию:
- **`initialize()`** — рукопожатие и **согласование версий** протокола;
- **`list_tools()`** — процедура **обнаружения (discovery)**: клиент узнаёт, какие инструменты есть, не зная о них заранее;
- **`call_tool(name, args)`** — вызов инструмента; результат приходит структурированным.

Весь код клиента асинхронный — используем `await`.

In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["mcp_shop_server.py"])

async def mcp_demo():
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()                 # согласование версий
            print("Сервер:", init.serverInfo.name, "| версия протокола:", init.protocolVersion)

            tools = (await session.list_tools()).tools         # discovery
            print("\nОбнаружены инструменты:")
            for t in tools:
                print(f"  - {t.name}: {t.description}")

            # Вызов обнаруженного инструмента
            res = await session.call_tool("search_products", {"query": "mug", "limit": 3})
            print("\nВызов search_products('mug'):")
            print(res.content[0].text)

await mcp_demo()

Сервер: shop | версия протокола: 2025-11-25

Обнаружены инструменты:
  - search_products: Найти товары по ключевому слову в названии (каталог на английском).
  - get_order: Получить статус заказа по его идентификатору.

Вызов search_products('mug'):
{
  "sku": "37410",
  "name": "Black And White Paisley Flower Mug",
  "price": 1.25
}


## Шаг 3. Мост MCP → языковая модель

Модель ожидает инструменты в формате OpenAI JSON Schema. MCP отдаёт схему каждого инструмента в поле `inputSchema`. Напишем конвертер и цикл агента, который:
1. получает список инструментов **из MCP** (discovery), а не из захардкоженного списка;
2. даёт их модели;
3. исполняет выбранные моделью вызовы **через MCP-сессию**;
4. возвращает результат модели.

Ключевая мысль: агент не знает заранее, какие инструменты существуют, — он узнаёт их у сервера в рантайме.

In [4]:
def mcp_to_openai(mcp_tools):
    """Преобразовать список MCP-инструментов в схемы формата OpenAI."""
    return [{"type": "function", "function": {
        "name": t.name,
        "description": t.description or "",
        "parameters": t.inputSchema,
    }} for t in mcp_tools]

async def run_mcp_agent(user_message: str, server: StdioServerParameters, max_steps: int = 5):
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools
            schemas = mcp_to_openai(mcp_tools)

            messages = [
                {"role": "system", "content":
                    "Ты — ассистент магазина. Инструменты подключены по MCP. "
                    "Каталог на английском: в query используй английские ключевые слова."},
                {"role": "user", "content": user_message},
            ]
            for step in range(max_steps):
                resp = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=schemas, max_tokens=800)
                msg = resp.choices[0].message
                messages.append(msg.model_dump())
                if not msg.tool_calls:
                    return msg.content
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"[шаг {step+1}] MCP call {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)  # исполнение через MCP
                    messages.append({"role": "tool", "tool_call_id": tc.id,
                                     "content": result.content[0].text})
            return "Достигнут лимит шагов."

answer = await run_mcp_agent("Проверь статус заказа ORD-1001", SERVER)
print("\n=== Ответ агента ===\n", answer)

[шаг 1] MCP call get_order({'order_id': 'ORD-1001'})



=== Ответ агента ===
 Вот информация по вашему заказу **ORD-1001**:

- **Товар:** White Hanging Heart T-Light Holder
- **SKU:** 85123A
- **Количество:** 2 шт.
- **Статус:** 🚚 **Отправлен** (shipped)

Ваш заказ уже в пути! Если нужна дополнительная информация — обращайтесь. 😊


## Шаг 4. Замена сервера без изменения агента

Главное преимущество MCP: сервер можно **заменить**, а код агента останется прежним. Рядом лежит второй сервер `mcp_shop_server_v2.py` — с тем же интерфейсом плюс **новый инструмент** `track_shipping`. Запустим того же агента (`run_mcp_agent`), лишь указав другой файл сервера: агент через discovery увидит новый инструмент и воспользуется им. Сначала покажем фрагмент v2 с новым инструментом:

In [5]:
from pathlib import Path
v2_src = Path("mcp_shop_server_v2.py").read_text()
# Покажем только фрагмент с новым инструментом track_shipping:
print(v2_src[v2_src.index("@mcp.tool()", v2_src.index("get_order")):])

@mcp.tool()
def track_shipping(tracking_code: str) -> dict:
    """НОВЫЙ инструмент: отследить доставку по трек-коду."""
    return TRACKING.get(tracking_code, {"error": "tracking_not_found"})


if __name__ == "__main__":
    mcp.run(transport="stdio")



In [6]:
SERVER_V2 = StdioServerParameters(command=sys.executable, args=["mcp_shop_server_v2.py"])

# Тот же агент, другой сервер. Код run_mcp_agent НЕ менялся.
answer = await run_mcp_agent(
    "Узнай статус заказа ORD-1001 и отследи его доставку", SERVER_V2)
print("\n=== Ответ агента (сервер v2) ===\n", answer)

[шаг 1] MCP call get_order({'order_id': 'ORD-1001'})


[шаг 2] MCP call track_shipping({'tracking_code': 'TRK-88'})



=== Ответ агента (сервер v2) ===
 📦 **Информация о заказе ORD-1001:**

- **Товар:** White Hanging Heart T-Light Holder (артикул 85123A)
- **Количество:** 2 шт.
- **Статус заказа:** Отправлен ✅

🚚 **Данные о доставке (TRK-88):**

- **Перевозчик:** RoyalMail
- **Состояние:** В пути
- **Ориентировочный срок прибытия:** 2 дня

Ваш заказ уже в пути и должен прибыть к вам в течение двух дней. Если нужны ещё какие-либо уточнения — обращайтесь!


## Шаг 5. Когда MCP — это overengineering

MCP оправдан, когда инструменты переиспользуются несколькими агентами/командами, меняются независимо от агента или должны быть заменяемы. Но у стандарта есть цена: отдельный процесс, сериализация, сетевой обмен, версии.

**MCP избыточен, если:**
- инструмент один и используется единственным агентом;
- нет требования подключать внешние/сторонние возможности;
- критична минимальная задержка, а инструмент — простая локальная функция.

Правило: начинайте с локальных инструментов (Тема 4); выносите в MCP-сервер тогда, когда появляется **повторное использование или независимая эволюция** инструментов.

## Итоги

- MCP снимает жёсткую связанность: инструменты живут на **сервере**, агент-**клиент** подключается к ним в рантайме.
- **`initialize`** согласует версии, **`list_tools`** реализует **discovery**, **`call_tool`** исполняет вызов; транспорт — JSON-RPC (здесь поверх stdio).
- Инструменты MCP легко **мостятся** в формат языковой модели — агент использует их так же, как локальные.
- Сервер можно **заменить** на расширенный **без правки кода агента** — мы добавили `track_shipping`, и агент подхватил его через discovery.
- Не применяйте MCP там, где хватает локальной функции.

**Дальше:** Тема 6 — память агента и RAG: как агент помнит пользователя и ищет знания в документах.